# Preheat sweep: 3D half-domain prediction panels

Six panels for the process-parameters section (`sec:params`): three held-out
steady preheat cases (highest preheat, smallest beam, biggest beam), ground
truth beside prediction, rendered as a half-domain metal block cut at the beam
plane -- the LP-FNO Fig. 5 view, in this manuscript's inferno convention.

The renderer here is a small local PyVista function rather than the demo's
`_render_iso_surface_img`: the demo has no half-domain three-quarter view, and
its files are not to be modified, so this notebook mirrors its mesh
construction (alpha >= 0.5 threshold with `all_scalars`, y-midplane clip) and
adds only the camera. All six panels share one colour limit so a colour means
the same temperature in every panel.

Model: `rigno_preheat107_cond_32k` (model 614f9c82), the RIGNO row of
`tab:preheat`. The predicted volumes are in `data/preheat_prediction_frames.npz`
(committed), from a local forward of that model whose input channels match its
evaluation preview bit for bit and whose prediction reproduces the stored one to
0.2 K mean in metal (`preheat_frames/local_forward.py`
in the code repository, checked 2026-08-28). The ground truth
is in `data/preheat_truth_frames.npz`, FLOW-3D output that the restricted record
holds, written by `prepare_truth_frames.py`.

In [ ]:
import os
import pathlib
import sys

import numpy as np

sys.path.insert(0, str(pathlib.Path.cwd()))
import figstyle

figstyle.apply()

PRED = pathlib.Path("data") / "preheat_prediction_frames.npz"     # committed
TRUTH = pathlib.Path("data") / "preheat_truth_frames.npz"         # restricted record: prepare_truth_frames.py

os.environ.setdefault("PYVISTA_OFF_SCREEN", "true")
import pyvista as pv  # noqa: E402

pv.OFF_SCREEN = True

MESH_UM = 10.0     # preheat60 grid: 61x40x50 cells at 10 um
MELT_ALPHA = 0.5   # metal threshold; GT alpha is Fraction Of Fluid

CASES = ("hot", "small", "big")

assert TRUTH.exists(), f"{TRUTH} missing: write it with prepare_truth_frames.py (restricted record)"
print(f"predictions: {PRED}; ground truth: {TRUTH}")

In [ ]:
def load_case(short):
    """Volumes in Kelvin (already scaled by the recorded ref_scales.T = 3000)."""
    with np.load(PRED) as zp, np.load(TRUTH) as zt:
        return dict(
            true_T=zt[f"{short}_true_T_K"],
            pred_T=zp[f"{short}_pred_T_K"],
            true_a=zt[f"{short}_true_alpha"],
            pred_a=zp[f"{short}_pred_alpha"],
            meta=zp[f"{short}_meta"],      # P [W], V [m/s], T0 [K], r [um]
            name=str(zp[f"{short}_name"]),
        )


_loaded = {short: load_case(short) for short in CASES}

# One colour range over the metal cells of ALL six volumes, so a colour means
# the same temperature in every panel and the peak-T deficit of the keyhole
# cases is visible rather than rescaled away.
hi = 0.0
for c in _loaded.values():
    for T, a in ((c["true_T"], c["true_a"]), (c["pred_T"], c["pred_a"])):
        metal = a >= MELT_ALPHA
        if np.any(metal):
            hi = max(hi, float(T[metal].max()))
GLOBAL_CLIM = [0.0, hi]

for short in CASES:
    c = _loaded[short]
    P, V, T0, r = c["meta"]
    print(f"{short:5s} {c['name']}: T0={T0:.0f} K r={r:.1f} um P={P:.0f} W V={V:.2f} m/s, "
          f"GT metal peak {c['true_T'][c['true_a'] >= MELT_ALPHA].max():.0f} K, "
          f"pred {c['pred_T'][c['pred_a'] >= MELT_ALPHA].max():.0f} K")
print(f"shared clim 0-{GLOBAL_CLIM[1]:.0f} K")

In [ ]:
import matplotlib.pyplot as plt

# Three-quarter camera on the half block: above-front-left, up +z, so the top
# face (keyhole depression), the beam-plane cut face and one end are all
# visible -- the LP-FNO Fig. 5 framing. Offsets in units of the longest edge.
CAM_OFFSET = (-0.06, -1.30, 1.50)
CAM_ZOOM = 1.25

PANEL_STEMS = {(short, tag): f"preheat3d_{short}_{tag}"
               for short in CASES for tag in ("gt", "pred")}


def _render_half_block(T, a, clim):
    """Half-domain metal block coloured by T, mirroring the demo's mesh build."""
    grid = pv.ImageData(dimensions=T.shape, spacing=(MESH_UM,) * 3)
    grid.point_data["alpha"] = np.asarray(a, np.float32).ravel(order="F")
    grid.point_data["T"] = np.asarray(T, np.float32).ravel(order="F")
    # all_scalars: keep only cells whose every corner is metal, so the surface
    # never carries gas-side values. Clip keeps the far half (y >= mid): the
    # exposed beam-plane cut face looks toward the -y camera.
    thr = grid.threshold(MELT_ALPHA, scalars="alpha", all_scalars=True)
    thr = thr.clip(normal=(0.0, 1.0, 0.0), origin=grid.center, invert=False)
    surf = thr.extract_surface()   # no smoothing: it would round the flat cut face
    plotter = pv.Plotter(off_screen=True, window_size=(1100, 825))
    try:
        plotter.set_background("white")
        plotter.add_mesh(surf, scalars="T", cmap=figstyle.FIELD_TEMPERATURE,
                         clim=list(clim), smooth_shading=True, ambient=0.25,
                         diffuse=0.75, show_scalar_bar=False)
        cx, cy, cz = grid.center
        L = max(T.shape) * MESH_UM
        dx, dy, dz = CAM_OFFSET
        plotter.camera_position = [(cx + dx * L, cy + dy * L, cz + dz * L),
                                   (cx, cy, cz), (0.0, 0.0, 1.0)]
        plotter.reset_camera()
        plotter.camera.zoom(CAM_ZOOM)
        plotter.reset_camera_clipping_range()
        return plotter.screenshot(return_img=True)
    finally:
        plotter.close()


def render_panel(short, tag, stem):
    """One half-domain block, no title: LaTeX supplies the subcaption."""
    c = _loaded[short]
    T, a = (c["true_T"], c["true_a"]) if tag == "gt" else (c["pred_T"], c["pred_a"])
    img = _render_half_block(T, a, GLOBAL_CLIM)
    fig = plt.figure(figsize=(5.2, 5.2 * img.shape[0] / img.shape[1]))
    fig.patch.set_facecolor("white")
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(img)
    ax.axis("off")
    return figstyle.save(fig, stem)


def render_cbar(stem="preheat3d_cbar"):
    return figstyle.colorbar_strip(stem, *GLOBAL_CLIM)


for (short, tag), stem in PANEL_STEMS.items():
    print(render_panel(short, tag, stem))
print(render_cbar())

## Camera explorer (manual use only)

Sliders re-render the big-beam ground-truth block so a new view can be found
by eye. The sliders touch nothing: to adopt a view, copy the printed
`CAM_OFFSET` / `CAM_ZOOM` line into the constants in the render cell above and
re-run the notebook with `LPNO_REPLACE_FIGURES=1`. Offsets are in units of the
longest block edge; each render takes a second or two, so the sliders only
fire on release.

In [ ]:
import os

if os.environ.get("LPNO_CAMERA_EXPLORER", "").strip() not in ("1", "true", "yes"):
    # Headless regeneration must not build widgets: `interact` under nbconvert
    # stalls the kernel waiting on widget comms (measured: the execute pass ran
    # past 500 s where the render cells alone take ~2 min). Opt in per run.
    print("camera explorer skipped; set LPNO_CAMERA_EXPLORER=1 and re-run this"
          " cell in a live notebook to use the sliders")
elif True:
    from ipywidgets import FloatSlider, interact
    _c_prev = _loaded["big"]

    def preview(dx, dy, dz, zoom):
        """Render one preview at the slider camera, leaving CAM_* untouched."""
        global CAM_OFFSET, CAM_ZOOM
        keep = (CAM_OFFSET, CAM_ZOOM)
        CAM_OFFSET, CAM_ZOOM = (dx, dy, dz), zoom
        try:
            img = _render_half_block(_c_prev["true_T"], _c_prev["true_a"], GLOBAL_CLIM)
        finally:
            CAM_OFFSET, CAM_ZOOM = keep
        fig = plt.figure(figsize=(7, 7 * img.shape[0] / img.shape[1]))
        ax = fig.add_axes([0, 0, 1, 1])
        ax.imshow(img)
        ax.axis("off")
        plt.show()
        print(f"CAM_OFFSET = ({dx:.2f}, {dy:.2f}, {dz:.2f})")
        print(f"CAM_ZOOM = {zoom:.2f}")

    interact(
        preview,
        dx=FloatSlider(value=CAM_OFFSET[0], min=-1.5, max=1.5, step=0.02,
                       description="yaw dx", continuous_update=False),
        dy=FloatSlider(value=CAM_OFFSET[1], min=-2.5, max=-0.5, step=0.02,
                       description="dist dy", continuous_update=False),
        dz=FloatSlider(value=CAM_OFFSET[2], min=0.2, max=2.5, step=0.02,
                       description="tilt dz", continuous_update=False),
        zoom=FloatSlider(value=CAM_ZOOM, min=0.8, max=1.8, step=0.01,
                         description="zoom", continuous_update=False),
    )